# 003: Soundness, Defect Coupling, and Quotient Geometry

**Local Green, Global Red** — Notebook 3 of 3

**Core Topic:** The Master Soundness Criterion, Theorem 5 (sufficient condition), Theorem 6 (why triviality is not necessary: structural coupling), and the Remark on quotient distance $\Phi(\omega) = \inf_t \|\omega - Dt\|_1$.

**AuthorityEffect:** None.


In [1]:
import sys
from pathlib import Path
from fractions import Fraction
import numpy as np

# Dynamically locate and import sibling code directory
nb_dir = Path.cwd().resolve()
candidates = [
    nb_dir.parent / "code",
    nb_dir / "code",
] + [p / "code" for p in nb_dir.parents if (p / "code" / "code_core.py").is_file()]

for c in candidates:
    if c.is_dir() and (c / "code_quotient_solver.py").is_file():
        if str(c) not in sys.path:
            sys.path.insert(0, str(c))
        break

from lgr_replay import (
    coupled_tallies, coupled_raw_norm, coupled_all_green_forces_zero,
    repair_mass, attained_at_zero, phi_lower_bound
)
from code_witnesses import (
    TrivialKernelThreePatchSystem, CoupledSystem, ScalarQuotientWitness
)
from code_theorems import (
    check_theorem_5_sufficiency, check_theorem_6_coupled_witness,
    check_remark_irreparable_red
)
from code_quotient_solver import QuotientDistanceSolver, MinkowskiNorm

print("[OK] Quotient solver and coupled theorem modules loaded successfully!")

[OK] Quotient solver and coupled theorem modules loaded successfully!


## 1. The Master Soundness Criterion

Let $D: C^0 \to C^1$ be the defect operator mapping an assignment $s$ to its interface defects:
$$\Delta(s) = Ds \in \prod_{e \in E} F(a)$$

The subspace of **realizable defects** is $\operatorname{im}(D)$.
The subspace of **undetectable defects** is the product of blind spots $\prod_e K_e$.

**Master Soundness Criterion.** All-Green forces Global Green ($I(s) = 0$) if and only if:
$$\operatorname{im}(D) \cap \prod_{e \in E} K_e = \{0\}$$

That is: *Soundness holds precisely when no nonzero realizable defect can hide inside the joint blind spots.*

## 2. Theorem 5: The Trivial Blind Spot Condition (Sufficient)

**Theorem 5 (Sufficient Condition).** If every interface is watched and has a trivial blind spot ($K_e = \{0\}$), then All-Green implies $I(s) = 0$.

*Reason:* If $K_e = \{0\}$ for all $e$, then the ambient product $\prod_e K_e = \{0\}$ is trivial.
Its intersection with $\operatorname{im}(D)$ is therefore automatically $\{0\}$, regardless of how sites are coupled!

Let's verify this condition on a system where each interface is watched by full-rank projections:

In [2]:
tk = TrivialKernelThreePatchSystem()
# Test on a zero assignment (All-Green and I=0)
from code_core import Assignment
zero_assignment = Assignment(states={name: np.zeros(3) for name in tk.system.sites})
assert check_theorem_5_sufficiency(tk.system, zero_assignment)
print("[PASSED] The displayed example checks passed: Trivial blind spots guarantee that All-Green forces I(s) = 0 on this system.")

[PASSED] The displayed example checks passed: Trivial blind spots guarantee that All-Green forces I(s) = 0 on this system.


## 3. Theorem 6: Triviality is Not Necessary (The Coupled System)

Must every interface have $K_e = \{0\}$ for the system to be sound? **No!**

**Theorem 6 (Coupling Restores Soundness).**

Take sites $(a_0,a_1,b)$, interfaces $e_i=(b,a_i)$, and zero translations. For every assignment,
$$
\Delta(s)_{e_i} = 0 \cdot s(a_i) - s(b) = -s(b).
$$
Thus both interface defects coincide without any equality assumption on $s(a_0)$ and $s(a_1)$.

To follow the Python coordinate convention, write $s(b)=(v_0,v_1)$. Evaluator 0 reads the second coordinate and evaluator 1 reads the first. Their scores are $(|v_1|,|v_0|)$, and their blind spots are $K_0=\mathbb R\times\{0\}$ and $K_1=\{0\}\times\mathbb R$. This is the paper's construction with the evaluator labels interchanged.

Both blind spots are nontrivial. Their product is a two-dimensional subspace of $(\mathbb R^2)^2$. All-Green forces $v_0=v_1=0$, hence both defects vanish and $I(s)=0$.

**Structural coupling compensates for individual blindness.**

In [3]:
samples = [
    (Fraction(0), Fraction(0)),
    (Fraction(1), Fraction(0)),
    (Fraction(0), Fraction(1)),
    (Fraction(3), Fraction(-2)),
    (Fraction(-7, 5), Fraction(11, 3)),
]

print(f"{'State v = (v1, v2)':>25} | {'Tallies (J0, J1)':>20} | {'True I(s)':>12} | {'All-Green => I=0?':>18}")
print("-" * 82)

for v in samples:
    tallies = coupled_tallies(v)
    raw_I = coupled_raw_norm(v)
    forces_zero = coupled_all_green_forces_zero(v)
    print(f"{str(v):>25} | {str(tallies):>20} | {str(raw_I):>12} | {str(forces_zero):>18}")
    assert forces_zero

cs = CoupledSystem()
assert check_theorem_6_coupled_witness(cs)
print("\n[PASSED] The displayed example checks passed: Even with 1D blind spots on every interface, All-Green forces I=0 on this system.")

       State v = (v1, v2) |     Tallies (J0, J1) |    True I(s) |  All-Green => I=0?
----------------------------------------------------------------------------------
(Fraction(0, 1), Fraction(0, 1)) | (Fraction(0, 1), Fraction(0, 1)) |            0 |               True
(Fraction(1, 1), Fraction(0, 1)) | (Fraction(0, 1), Fraction(1, 1)) |            2 |               True
(Fraction(0, 1), Fraction(1, 1)) | (Fraction(1, 1), Fraction(0, 1)) |            2 |               True
(Fraction(3, 1), Fraction(-2, 1)) | (Fraction(2, 1), Fraction(3, 1)) |           10 |               True
(Fraction(-7, 5), Fraction(11, 3)) | (Fraction(11, 3), Fraction(7, 5)) |       152/15 |               True

[PASSED] The displayed example checks passed: Even with 1D blind spots on every interface, All-Green forces I=0 on this system.


## 4. The Remark: Raw Inconsistency vs Irreparable Quotient Defect

The paper highlights two fundamentally distinct kinds of inconsistency:
1. **Raw Inconsistency $I(s) = \sum_e \|\Delta(s)_e\|_1$:**
   Disagreement that is generated by an actual section $s$. In the 3-patch system, $I(s_R) = R > 0$, but the defect is exact (it lies in $\operatorname{im} D$).
2. **Quotient Residual $\Phi(\omega) = \inf_{t \in C^0} \|\omega - Dt\|_1$:**
   The infimal residual distance under any possible internal adjustment of site states $t$.
   If $\Phi(\omega) > 0$, the discrepancy **cannot be repaired within the image of the fixed repair operator D**.

### The Irreparable Cochain $\omega = (1, -1)$
On the coupled system with scalar states, $Dt = (-t, -t)$ has diagonal image.
Consider the cochain $\omega = (1, -1)$:
$$\Phi(\omega) = \inf_{t \in \mathbb{R}} (|1 + t| + |-1 + t|) = 2 > 0$$
The minimum is attained at $t = 0$.

Let's use the Linear Programming quotient solver to estimate this using floating-point linear programming:

In [4]:
# Define coboundary matrix D for the scalar coupled system:
# D * t = [-1; -1] * t
coboundary_D = np.array([[-1.0], [-1.0]])

# Test cochain omega = [1.0, -1.0]
omega = np.array([1.0, -1.0])

# Solve quotient distance using floating-point linear programming
res = QuotientDistanceSolver.solve(
    omega=omega,
    D=coboundary_D,
    norm=MinkowskiNorm.L1
)

print(f"Optimal quotient distance Phi(omega): {res.primal_distance:.6f}")
print(f"Optimal site adjustment t*:          {res.optimal_alpha}")
print(f"Residual cochain (omega - D*t*):     {res.residual}")
print(f"Dual witness certificate w:          {res.dual_witness}")
print(f"Dual objective:                      {res.dual_objective:.6f}")
print(f"Duality gap:                         {res.duality_gap:.2e}")
print(f"Is numerically certified:            {res.is_numerically_certified}")

assert abs(res.primal_distance - 2.0) < 1e-6
assert res.is_numerically_certified
assert attained_at_zero()
assert phi_lower_bound() == Fraction(2)
assert check_remark_irreparable_red()

print("\n[CONFIRMED] The numerical estimate agrees with the exact analytic distance 2.")

Optimal quotient distance Phi(omega): 2.000000
Optimal site adjustment t*:          [1.]
Residual cochain (omega - D*t*):     [2. 0.]
Dual witness certificate w:          [ 1. -1.]
Dual objective:                      2.000000
Duality gap:                         4.44e-16
Is numerically certified:            True

[CONFIRMED] The numerical estimate agrees with the exact analytic distance 2.


### Final Synthesis: The Core Lessons of Local Green, Global Red
1. **Passivity:** Evaluators can never manufacture inconsistency ($J^\alpha \le I$).
2. **The Mechanism of False Green:** A defect is invisible exactly when it lies in the corresponding joint kernel.
3. **Coverage $\ne$ Soundness:** A nonzero realizable defect in the product of watcher kernels yields arbitrarily large invisible defect mass by scaling, when assignments are unrestricted.
4. **Coupling Can Restore Soundness:** Soundness requires $\operatorname{im}(D) \cap \prod_e K_e = \{0\}$; structural coupling can eliminate unobserved directions.
5. **Raw vs Quotient:** A raw inconsistency might be reparable by internal realignment (exact cochain), while an irreparable defect ($\Phi > 0$) represents an insurmountable structural obstruction.